# Session 10 — XGBoost with scale_pos_weight

Per `docs/Credit_Risk_Pipeline_Plan_v3.md` (buổi 10).

**Goal**: train XGBoost for both feature sets (`portfolio`, `at_application`), using
`scale_pos_weight` computed from the class ratio (no SMOTE — that's session 11's
comparison), and compare against session 9's Logistic Regression baselines.

Reuses `model/preprocessing.py::build_pipeline` unchanged — same `ColumnTransformer`,
different `estimator` argument. That parameterization was added specifically so this
notebook doesn't need its own copy of the `StandardScaler`/`OneHotEncoder` setup.

In [1]:
import sys

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

sys.path.insert(0, '../etl')
sys.path.insert(0, '..')
from config import get_engine
from model.features import (
    TARGET_COL,
    get_feature_columns,
    split_numeric_categorical,
)
from model.preprocessing import build_pipeline

engine = get_engine()

## Load data

Same `ml_features` view session 9 used — 25 columns, already filtered to `data_source = 'historical'`.

In [2]:
df = pd.read_sql("SELECT * FROM ml_features", engine)
print(df.shape)

(32581, 25)


## Feature sets and split

Identical to session 9: same two feature sets, same stratified 80/20 split with
`random_state=42`. Using the same split (not a fresh one) means any change in the
comparison table is attributable to the model, not to a different train/test partition.

In [3]:
feature_cols = {
    "portfolio": get_feature_columns(df.columns, "portfolio"),
    "at_application": get_feature_columns(df.columns, "at_application"),
}

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)
print(f"train: {train_df.shape}, test: {test_df.shape}")

train: (26064, 25), test: (6517, 25)


## Expected result — written before running

Per `REVIEW.md`'s pre-session-10 review: session 8's scan found 9 of 22 features are
statistically indistinguishable from noise (`past_delinquencies` single-feature AUC
0.5004, `open_accounts` 0.4980, `credit_utilization_ratio` 0.5051). That caps what any
model — including XGBoost — can extract from this dataset. Writing the expectation down
*before* running is the only way it can actually catch a bug instead of being explained
away after the fact.

**Expected**: `at_application` ROC-AUC roughly **0.82-0.84** (session 9's Logistic
Regression baseline was 0.8072) — a modest gain from XGBoost's ability to model
interactions and non-linearities, not a leap. PR-AUC should move by a comparable or
slightly larger margin, consistent with session 9's finding that PR-AUC is the more
sensitive metric on this imbalanced target.

**If the result is ROC-AUC > 0.90**: stop and look for a bug or a reintroduced leakage
column before trusting it. A dataset with 9 noise columns and no new information doesn't
suddenly support a near-perfect model just because the algorithm changed.

## `scale_pos_weight`

`scale_pos_weight = count(negative) / count(positive)`, computed on the **train split
only** — the same fit-on-train-only discipline as session 9's `StandardScaler`. Computing
it on the full dataset would leak the test set's exact class balance into a training
hyperparameter, which is a smaller version of the same leak fitting a scaler on the full
data would cause.

XGBoost uses this to upweight the minority (default) class in the loss, the boosting
equivalent of `class_weight="balanced"` in session 9's Logistic Regression — this
notebook does **not** use SMOTE (oversampling by synthesizing new minority rows).
Session 11 evaluates SMOTE against `scale_pos_weight` directly and picks the better one
by PR-AUC; doing that comparison here would be premature without today's baseline number
to compare it to.

In [4]:
neg, pos = train_df[TARGET_COL].value_counts().sort_index()
scale_pos_weight = neg / pos
print(f"train class counts: negative={neg}, positive={pos}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

train class counts: negative=20378, positive=5686
scale_pos_weight = 3.5839


## Train + evaluate both feature sets

In [5]:
results = {}

for name, cols in feature_cols.items():
    numeric_cols, categorical_cols = split_numeric_categorical(cols)
    estimator = XGBClassifier(
        # These three are untuned starting values, chosen before running anything and
        # left alone afterwards - not the output of a search. 300 shallow trees at a low
        # learning rate is a conventional starting point for a tabular set this size
        # (~26k training rows, 35 columns after one-hot), nothing more specific than
        # that. No grid/random search was run, so there is no evidence these beat any
        # other reasonable triple; hyperparameter tuning is future work, not something
        # this notebook did. Stated plainly because writing a rationale now - with the
        # result already known - would be the same post-hoc storytelling the "Expected
        # result" cell above exists to prevent.
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
    )
    pipeline = build_pipeline(numeric_cols, categorical_cols, estimator=estimator)
    pipeline.fit(train_df[cols], train_df[TARGET_COL])

    proba = pipeline.predict_proba(test_df[cols])[:, 1]
    results[name] = {
        "roc_auc": roc_auc_score(test_df[TARGET_COL], proba),
        "pr_auc": average_precision_score(test_df[TARGET_COL], proba),
    }

xgb_results = pd.DataFrame(results).T
xgb_results

,roc_auc,pr_auc
portfolio,0.937225,0.884643
at_application,0.890672,0.805474


## Comparison: Logistic Regression baseline vs XGBoost

Session 9's baseline numbers, hardcoded here rather than re-derived, so this table
compares against the exact numbers already reported and reviewed — not a re-run that
could silently drift if the data or split changed.

In [6]:
# PINNED, not re-derived: these are 02_baseline_model.ipynb's outputs as of commit
# f1cb74c (2026-09-12, "feat: complete session 9"). Hardcoding avoids silent drift from a
# re-run, but it also means they can go stale - if session 9's notebook is re-executed
# with different data, a different split, or a changed feature set, these numbers are
# wrong and nothing here will notice. Re-check against 02_baseline_model.ipynb's final
# cell before trusting this comparison.
baseline_results = pd.DataFrame({
    "portfolio": {"roc_auc": 0.871252, "pr_auc": 0.720649},
    "at_application": {"roc_auc": 0.807215, "pr_auc": 0.623982},
}).T

comparison = pd.concat(
    {"logistic_regression": baseline_results, "xgboost": xgb_results},
    axis=1,
)
comparison

logistic_regression             xgboost          
                           roc_auc    pr_auc   roc_auc    pr_auc
portfolio                 0.871252  0.720649  0.937225  0.884643
at_application            0.807215  0.623982  0.890672  0.805474

## Reading the comparison

**The written expectation (0.82-0.84 ROC-AUC for `at_application`) was wrong — the actual
result is 0.8907.** That's a bigger gain than predicted, and the reason to write an
expectation down first is to catch exactly this rather than narrate it as expected
afterwards. Checked before trusting it:

1. **No leakage.** Re-derived `at_application`'s column list directly and confirmed
   `loan_grade`/`loan_int_rate` are absent — the same 18 columns the pipeline used.
2. **Mild overfitting, not a bug.** Train ROC-AUC 0.9178 vs test 0.8907 — a modest gap
   for a 300-tree, depth-4 ensemble on ~26k rows.
3. **Cross-validated** — see the correction below, because the first attempt at this
   check was itself wrong.

> ### ⚠️ Correction (made in session 11)
>
> This cell originally reported 5-fold CV at **0.8606 ± 0.0355** and concluded the holdout
> number "sits on the optimistic side of a wide distribution", with variance ~5× the
> Logistic Regression baseline. That used `cross_val_score(..., cv=5)`, which splits
> **without shuffling**, and the rows of `ml_features` are not randomly ordered with
> respect to the target — default rate across five contiguous blocks runs 27.8%, 19.1%,
> 24.3%, 18.0%, 20.0%. The spread was measuring the table's row order, not the model.
>
> With `StratifiedKFold(shuffle=True, random_state=42)`: **0.8923 ± 0.0048**.
>
> So the holdout 0.8907 agrees with the CV mean to within 0.002 — it was never optimistic
> — and XGBoost's spread (±0.0048) is *tighter* than the LR baseline's (±0.0083), not 5×
> wider. See `04_smote_comparison.ipynb` for the full write-up.

**Revised read**: XGBoost genuinely and stably outperforms the linear baseline here
(0.8923 vs 0.8062 shuffled CV ROC-AUC on the same feature set). The gain comes from
modelling non-linear interactions among the real signal columns (`loan_percent_income`,
`debt_to_income_ratio`, `income`, `emp_length`) that a linear model can only capture via
hand-built cross terms — session 12's SHAP analysis is how to confirm that rather than
assume it.

The `portfolio` vs `at_application` gap (0.9372 vs 0.8907 = 0.047 ROC-AUC) is narrower in
relative terms than session 9's (0.064 on a lower base), consistent with XGBoost partly
reconstructing what `loan_grade` encoded from correlated features. Worth naming explicitly
in an interview: a tree model closing part of the leakage gap is **not** evidence the
leakage columns are safe to use — it's evidence that removing a feature doesn't remove the
information if correlated substitutes remain.